# Notebook 02: DeepHAM for Krusell–Smith: learning a generalized moment

**Course:** Summer School on AI for Economics and Finance · ESOMAS, University of Torino (August 24–26, 2026)
**Session:** Day 1, 15:30 – 17:00: Solving Heterogeneous Agent Models with DeepHAM and Structural Reinforcement Learning
**Slides:** `../../slides/DeepHAM_Lecture_slides.pdf`
**Notebook role:** core (in-class walkthrough)
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/summer-school-AI-for-economics-and-finance-2026)

---


Same model, same algorithm, same code — **one line of configuration changes**. Instead of
summarising the wealth distribution by its mean, we let the algorithm learn its own
summary statistic (`n_fm = 0`, `n_gm = 1`):

$$Q_t \;=\; \frac{1}{N}\sum_{i=1}^{N} \mathcal{Q}\big(a^i_t\big),$$

where the basis function $\mathcal{Q}$ is a small neural network trained *jointly* with the
value function. If $\mathcal{Q}$ were the identity, $Q_t$ would be exactly aggregate capital
and we would be back in notebook 01. It is not the identity, and notebook 05 shows what it
learns instead.

Each value network carries its own basis $\mathcal{Q}_i$ (`ValueTrainer.gm_model`), and the
policy network carries a separate one. They are saved alongside the network weights as
`value{i}_gm.weights.h5` and `policy_gm.weights.h5`.

In [ ]:
RUN_MODE = "smoke"   # one of: "smoke", "teaching", "production"
SEED_INDEX = 3       # which entry of config["random_seed"] to use

## 1. Move into the code directory

DeepHAM is a package of plain Python modules (`param.py`, `dataset.py`, `value.py`,
`policy.py`, ...) that expect to be imported with `src/` as the working directory, with the
data alongside it in `../data`.

On Nuvolos the notebook server starts in `/files`, which mirrors the course repository, so
the path below is the same one you see on GitHub. If you are running from a local clone or
from Colab instead, just open the notebook from inside `DeepHAM_nuvolos/src` and the cell
leaves the working directory alone.

In [ ]:
import os
import sys

# On Nuvolos the kernel starts in /files, which mirrors the course repository.
NUVOLOS_SRC = "/files/day1/Yang/code/DeepHAM_nuvolos/src"
if os.path.isdir(NUVOLOS_SRC):
    os.chdir(NUVOLOS_SRC)

# Anywhere else (local clone, Colab) the notebook's own folder is already src/.
if not os.path.isfile("param.py"):
    raise FileNotFoundError(
        f"Expected to be in DeepHAM_nuvolos/src, but the working directory is "
        f"{os.getcwd()!r}. On Nuvolos that is {NUVOLOS_SRC!r}; elsewhere, open this "
        f"notebook from inside src/ or os.chdir() there yourself."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

In [ ]:
import json
import time
import datetime

import numpy as np

from param import KSParam
from dataset import KSInitDataSet
from value import ValueTrainer
from policy import KSPolicyTrainer
from simulation_KS import simul_shocks, simul_k
from util import print_elapsedtime
from util import set_random_seed

## 2. Choose the run mode

Everything expensive in DeepHAM is controlled by a handful of numbers. The cell below maps
`RUN_MODE` onto them, so the notebook can be run end to end in a few minutes during class
and at the published setting afterwards.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| policy gradient steps | 100 | 1,500 | 10,000 |
| unroll horizon $T$ | 60 | 150 | 150 |
| simulated paths | 64 | 192 | 384 |
| value-net epochs | 10 | 60 | 200 |
| measured wall clock | 2.2 min | 15.6 min | ~90 min |
| mean capital reached | ~11 | ~32 | ~39 (the KS level) |

Timings are measured on a Colab A100; a Nuvolos CPU is slower, though most of the wall
clock is the NumPy simulation rather than the networks. The capital row is the honest
summary of what each budget buys: `smoke` exercises every code path but converges to
nothing, `teaching` gets most of the way to the Krusell–Smith level of $K pprox 39$, and
`production` reproduces the published results (the reference run shipped in
`../data/simul_results` took 5,278 s). Two invariants are asserted rather than
left implicit, because violating either fails deep inside the training loop with an opaque
shape error:

* `policy_config["valid_size"] == dataset_config["n_path"]` — the fixed validation batch is
  built from `init_ds.datadict`, while its shocks are simulated with `valid_size` rows.
* `policy_config["batch_size"] <= policy_config["valid_size"]` — asserted inside
  `PolicyTrainer.train`.

In [ ]:
CONFIG_PATH = "./configs/KS/game_nn_n50_0fm1gm.json"   # 0 fixed moments, 1 learned generalized moment
EXP_NAME = "1gm"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

seed = config["random_seed"][SEED_INDEX]
set_random_seed(seed)
print(f"Solving {CONFIG_PATH} with seed {seed} (index {SEED_INDEX})")
print(
    f'n_fm = {config["n_fm"]} fixed moment(s), '
    f'n_gm = {config["n_gm"]} generalized moment(s), '
    f'{config["n_agt"]} agents'
)

In [ ]:
# Training budget, dispatched on RUN_MODE (see the run-mode cell above).
if RUN_MODE == "smoke":            # exercises every code path, converges to nothing much
    N_PATH, T_BURN = 64, 300
    V_T, V_COUNT, V_EPOCH = 700, 400, 10
    NUM_STEP, T_UNROLL = 100, 60
    FREQ_VALID, FREQ_UPDATE_V = 50, 50
elif RUN_MODE == "teaching":       # gets most of the way to the KS capital stock
    N_PATH, T_BURN = 192, 2000
    V_T, V_COUNT, V_EPOCH = 1200, 600, 60
    NUM_STEP, T_UNROLL = 1500, 150
    FREQ_VALID, FREQ_UPDATE_V = 250, 500
elif RUN_MODE == "production":     # the setting behind the published results
    N_PATH, T_BURN = 384, 6000
    V_T, V_COUNT, V_EPOCH = 2000, 800, 200
    NUM_STEP, T_UNROLL = 10000, 150
    FREQ_VALID, FREQ_UPDATE_V = 500, 2000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

config["dataset_config"]["n_path"] = N_PATH
config["dataset_config"]["t_burn"] = T_BURN
config["value_config"]["T"] = V_T
config["value_config"]["t_count"] = V_COUNT
config["value_config"]["num_epoch"] = V_EPOCH
config["policy_config"]["num_step"] = NUM_STEP
config["policy_config"]["t_unroll"] = T_UNROLL
config["policy_config"]["freq_valid"] = FREQ_VALID
config["policy_config"]["freq_update_v"] = FREQ_UPDATE_V
# valid_size must match n_path: the validation batch comes from init_ds.datadict
# (n_path rows) while its shocks are simulated with valid_size rows.
config["policy_config"]["valid_size"] = N_PATH
config["policy_config"]["batch_size"] = min(config["policy_config"]["batch_size"], N_PATH)
config["value_config"]["batch_size"] = min(config["value_config"]["batch_size"], N_PATH)

assert config["policy_config"]["valid_size"] == config["dataset_config"]["n_path"]
assert config["policy_config"]["batch_size"] <= config["policy_config"]["valid_size"]
assert config["value_config"]["t_count"] < config["value_config"]["T"] - 1, \
    "t_count must leave at least one time slice inside the value simulation"
assert config["policy_config"]["num_step"] >= config["policy_config"]["freq_valid"], \
    "num_step < freq_valid gives zero training epochs (n_epoch = num_step // freq_valid)"

print(
    f"RUN_MODE={RUN_MODE}: {NUM_STEP} policy steps, unroll {T_UNROLL}, "
    f"{N_PATH} paths, {V_EPOCH} value-net epochs"
)

### Where the results go

In [ ]:
mparam = KSParam(config["n_agt"], config["beta"], config["mats_path"])

# The run mode is part of the directory name, so a quick smoke run can never overwrite a
# long production run -- and neither can overwrite the reference solutions shipped in the
# repository (game_nn_n50_1fm1, game_nn_n50_1gm3).
model_path = "../data/simul_results/KS/game_{}_n{}_{}_{}".format(
    config["dataset_config"]["value_sampling"], config["n_agt"], EXP_NAME, RUN_MODE
)
config["model_path"] = model_path
config["current_time"] = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs(model_path, exist_ok=True)
with open(os.path.join(model_path, "config_beg.json"), "w") as f:
    json.dump(config, f)
print("Results will be written to", model_path)

## 3. Build the initial dataset

`KSInitDataSet` first burns in a panel of `n_path` economies of `n_agt` agents under the
Krusell–Smith benchmark policy, so that the cross-section of wealth starts from its
ergodic distribution. The value networks are then trained on data simulated from an
*initial* policy: either that same benchmark policy (`init_with_bchmk = true`) or a
constant consumption share (`init_with_bchmk = false`, the setting used here).

In [ ]:
start_time = time.monotonic()

init_ds = KSInitDataSet(mparam, config)
value_config = config["value_config"]

if config["init_with_bchmk"]:
    init_policy = init_ds.k_policy_bchmk        # the KS benchmark (b-spline) policy
    policy_type = "pde"
else:
    init_policy = init_ds.c_policy_const_share  # a constant consumption share
    policy_type = "nn_share"

# Supervised targets for the value nets: discounted utility along simulated paths.
train_vds, valid_vds = init_ds.get_valuedataset(init_policy, policy_type, update_init=False)
print_elapsedtime(time.monotonic() - start_time)

## 4. Pre-train the value networks

`num_vnet` independent value networks are fitted to the same targets. They are used as the
terminal bootstrap $\beta^{T}V(s_T)$ that closes the finite unroll in the policy objective;
averaging several of them reduces the variance of that bootstrap.

Because `n_gm = 1`, each `ValueTrainer` also builds a `gm_model`: a two-layer network
(`gm_config["net_width"] = [12, 12]`, `tanh`) mapping one normalised asset holding to one
number. `GeneralizedMomModel.call` applies it agent by agent and averages, so the moment is
permutation-invariant by construction — the value function cannot depend on *which* agent
holds what, only on the distribution.

In [ ]:
vtrainers = []
for i in range(value_config["num_vnet"]):
    config["vnet_idx"] = str(i)
    vtrainers.append(ValueTrainer(config))

for i, vtr in enumerate(vtrainers):
    print(f"--- value net {i} ---")
    vtr.train(train_vds, valid_vds, value_config["num_epoch"], value_config["batch_size"])

## 5. Train the policy, refreshing the value functions as it goes

`KSPolicyTrainer.train` runs the outer loop:

* **every step** — draw a mini-batch of states from `policy_ds`, simulate fresh shocks, and
  take one policy-gradient step on the unrolled objective;
* **every `freq_valid` steps** — evaluate on a fixed validation batch and print the mean
  discounted utility;
* **every `freq_update_v` steps** — rebuild the value dataset *under the current policy* and
  retrain the value nets, then hard-refresh the policy dataset (including its normalisation
  statistics) on the next rebuild.

That last loop is what makes this a fixed-point iteration rather than a one-shot fit: the
value functions that anchor the unroll are themselves re-estimated under the policy being
learned.

In [ ]:
policy_config = config["policy_config"]
ptrainer = KSPolicyTrainer(vtrainers, init_ds)
ptrainer.train(policy_config["num_step"], policy_config["batch_size"])

## 6. Save, and check that the run produced something sane

In [ ]:
with open(os.path.join(model_path, "config.json"), "w") as f:
    json.dump(config, f)

for i, vtr in enumerate(vtrainers):
    vtr.save_model(os.path.join(model_path, "value{}.weights.h5".format(i)))
ptrainer.save_model(os.path.join(model_path, "policy.weights.h5"))

elapsed = time.monotonic() - start_time
with open(os.path.join(model_path, "time.txt"), "w") as f:
    f.write(f"{CONFIG_PATH} at RUN_MODE={RUN_MODE} took {elapsed:.2f} seconds.\n")

print_elapsedtime(elapsed)
print("Saved to", model_path)

In [ ]:
# End-to-end check: the saved policy should simulate to a finite, economically plausible
# capital stock. This is a sanity check on the pipeline, not on convergence -- a `smoke`
# run is far too short to be accurate.
for fname in ["policy.weights.h5", "config.json", "stats.json"]:
    assert os.path.exists(os.path.join(model_path, fname)), f"missing {fname} in {model_path}"

_state = init_ds.next_batch(16)
_shocks = simul_shocks(16, 50, mparam, _state)
_sim = simul_k(
    16, 50, mparam, ptrainer.current_c_policy,
    policy_type="nn_share", state_init=_state, shocks=_shocks,
)
_K = _sim["k_cross"].mean()
assert np.isfinite(_K) and 1.0 < _K < 500.0, f"implausible mean capital {_K}"
print(f"Check passed: mean capital over a short simulation = {_K:.2f}")

## Summary

* One configuration change (`n_fm: 1, n_gm: 0` → `n_fm: 0, n_gm: 1`) swaps a hand-picked
  moment for a learned one. No other code changed.
* The basis function is trained by the same gradients that train the value function: it is
  optimised *for the decision problem*, not to fit the distribution.
* Averaging over agents is what makes the representation scale — the same network works for
  any number of agents.

## Takeaway

Table 2 of the paper reports that this substitution cuts the Bellman-equation error by
roughly two thirds relative to the mean-only baseline of notebook 01. Notebook 05 opens up
the learned $\mathcal{Q}$ and asks what it actually is.